<a href="https://colab.research.google.com/github/Munky213/termux-app/blob/master/notebook37b319e381.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

arc_prize_2026_arc_agi_3_path = kagglehub.competition_download('arc-prize-2026-arc-agi-3')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  ARC Prize 2026 – ARC-AGI-3  |  DEQGT × ANJ Agent  v10 ONE-SHOT FINAL  ║
# ║  ONE CELL. PASTE. SAVE VERSION → SAVE & RUN ALL. SUBMIT.                ║
# ║  Emmy Noether conservation · Kepler ratios · Tesla full-loop energy      ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# ══════════════════════════════════════════════════════════════════════════
# ▶▶▶  PARQUET WRITTEN ON LINE 1 — Kaggle sees it even if later code fails  ◀◀◀
# ══════════════════════════════════════════════════════════════════════════
import os, sys, json, math, hashlib, shutil, time
from pathlib import Path
import numpy as np
import pandas as pd

WORK    = Path("/kaggle/working")
PARQUET = WORK / "submission.parquet"
WORK.mkdir(exist_ok=True)

pd.DataFrame({
    "row_id":      ["boot_0"],
    "game_id":     ["boot"],
    "end_of_game": [True],
    "score":       [1],
}).to_parquet(str(PARQUET), index=False)
print(f"[BOOT] parquet stub written → {PARQUET.stat().st_size} bytes  ✅")

# ══════════════════════════════════════════════════════════════════════════
# PATHS & CONFIG
# ══════════════════════════════════════════════════════════════════════════
COMP      = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3")
WHEELS    = COMP / "arc_agi_3_wheels"
AGENTS_IN = COMP / "ARC-AGI-3-Agents"
AGENTS_WK = WORK / "ARC-AGI-3-Agents"
TRACES    = WORK / "traces"

IS_RERUN = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN", ""))

GAME_IDS = [
    "ar25","bp35","cd82","cn04","dc22","ft09","g50t",
    "ka59","lf52","lp85","ls20","m0r0","r11l","re86",
    "s5i5","sb26","sc25","sk48","sp80","su15","tn36",
    "tr87","tu93","vc33","wa30",
]

print("═"*65)
print("  ARC-AGI-3 | DEQGT × ANJ Agent v10 ONE-SHOT FINAL")
print(f"  Mode : {'COMPETITION RERUN ← GATEWAY ACTIVE' if IS_RERUN else 'OFFLINE / DRY-RUN'}")
print(f"  Games: {len(GAME_IDS)} public  |  110 private (evaluation)")
print("═"*65)

# ══════════════════════════════════════════════════════════════════════════
# [1] INSTALL WHEELS
# ══════════════════════════════════════════════════════════════════════════
print("\n[1/9] Installing arc-agi wheels …")
rc = os.system(
    f"pip install --no-index --find-links {WHEELS} arc-agi python-dotenv -q"
)
print(f"      pip exit={rc}")

try:
    import arcengine as _ae
    from arcengine import FrameData, GameAction, GameState
    HAS_AE = True
    print(f"      arcengine {getattr(_ae,'__version__','?')} ✅")
except ImportError as e:
    HAS_AE = False
    print(f"      [WARN] arcengine not available ({e}) — using offline stubs")

# ══════════════════════════════════════════════════════════════════════════
# [2] COPY FRAMEWORK
# ══════════════════════════════════════════════════════════════════════════
print("\n[2/9] Copying ARC-AGI-3-Agents framework …")
if AGENTS_IN.exists():
    shutil.copytree(str(AGENTS_IN), str(AGENTS_WK), dirs_exist_ok=True)
    print(f"      Copied → {AGENTS_WK}")
else:
    AGENTS_WK.mkdir(parents=True, exist_ok=True)
    print(f"      [WARN] {AGENTS_IN} not found — created empty dir")

# ══════════════════════════════════════════════════════════════════════════
# [3] PATCH __init__.py IMMEDIATELY — kills langgraph/smolagents crash
#     Must happen BEFORE sys.path.insert + BEFORE any `from agents…` import
# ══════════════════════════════════════════════════════════════════════════
print("\n[3/9] Patching agents/__init__.py (kill langgraph/smolagents) …")
_agents_pkg = AGENTS_WK / "agents"
_agents_pkg.mkdir(parents=True, exist_ok=True)
_init_path = _agents_pkg / "__init__.py"
_init_path.write_text("# safe stub — overwritten after Agent is loaded\n", encoding="utf-8")
print(f"      {_init_path}  →  safe stub ✅")

# ══════════════════════════════════════════════════════════════════════════
# [4] sys.path + import real Agent base
# ══════════════════════════════════════════════════════════════════════════
print("\n[4/9] sys.path + Agent base import …")
if str(AGENTS_WK) not in sys.path:
    sys.path.insert(0, str(AGENTS_WK))
print(f"      sys.path[0] = {AGENTS_WK}")

try:
    from agents.agent import Agent as _AgentBase
    _REAL_BASE = True
    print("      Agent base : REAL agents.agent.Agent ✅")
except Exception as _e:
    _REAL_BASE = False
    print(f"      Agent base : inline stub ({_e})")
    class _AgentBase:
        def __init__(self, game_id, *a, **kw):
            self.game_id        = str(game_id)
            self.action_counter = 0

# ══════════════════════════════════════════════════════════════════════════
# [5] arcengine offline stubs (only used when wheels unavailable)
# ══════════════════════════════════════════════════════════════════════════
if not HAS_AE:
    print("\n[5/9] Building arcengine stubs for offline mode …")
    class GameState:
        NOT_PLAYED   = "NOT_PLAYED"
        NOT_FINISHED = "NOT_FINISHED"
        WIN          = "WIN"
        GAME_OVER    = "GAME_OVER"

    class GameAction:
        _REG: dict = {}
        def __init__(self, n, v):
            self.name = n;  self.value = v
            self.reasoning = None;  self._data = {}
        def is_complex(self):   return self.name == "ACTION6"
        def set_data(self, d):  self._data = d
        @classmethod
        def from_name(cls, n):
            n = n.upper()
            if n not in cls._REG:
                raise ValueError(f"Unknown action: {n}")
            return cls._REG[n]

    for _n, _v in [("RESET",0),("ACTION1",1),("ACTION2",2),("ACTION3",3),
                   ("ACTION4",4),("ACTION5",5),("ACTION6",6),("ACTION7",7)]:
        _obj = GameAction(_n, _v)
        GameAction._REG[_n] = _obj
        setattr(GameAction, _n, _obj)

    class FrameData:
        def __init__(self, grid, state, levels_completed, available_actions):
            self.frame              = [grid]
            self.state              = state
            self.levels_completed   = levels_completed
            self.available_actions  = available_actions
    print("      arcengine stubs ready ✅")
else:
    print("\n[5/9] arcengine already loaded — stubs skipped")

# ══════════════════════════════════════════════════════════════════════════
# [6] DEFINE MyAgent (DEQGT × ANJ — integer-only, full-loop, no decimals)
#     OBSERVE → ASSIMILATE → EMULATE → TEACH
#     Fibonacci-Prime harmonics · Bijective T/T⁻¹ · Global Harmonic Memory
# ══════════════════════════════════════════════════════════════════════════
print("\n[6/9] Defining MyAgent (DEQGT × ANJ v10) …")

# ── ANJ constants (integer-native, Fibonacci-Prime sequence) ──────────────
_FIB       = [1, 2, 3, 5, 8, 13, 21, 34, 55]
_MAX_GAME  = 300
_PROBE_CAP = 8
_ALL_ACTS  = ["ACTION1","ACTION2","ACTION3","ACTION4","ACTION5","ACTION6","ACTION7"]

def _grid(fd) -> np.ndarray:
    raw = fd.frame[-1] if fd.frame else []
    if not raw:
        return np.zeros((1,1), dtype=np.int32)
    return np.clip(np.array(raw, dtype=np.int32), 0, 15)

def _state(fd) -> str:
    st = fd.state
    return st.name if hasattr(st, "name") else str(st)

def _enc(g: np.ndarray, n: int = 9) -> np.ndarray:
    """Phase-harmonic signature — integer-scaled ×1000, no decimals stored."""
    if g.size == 0:
        return np.zeros(n * 2, dtype=np.int64)
    t = g.ravel().astype(float) * (2 * math.pi / 16)
    sig = []
    for k in _FIB[:n]:
        c = np.mean(np.exp(1j * k * t))
        sig += [int(round(c.real * 1000)), int(round(c.imag * 1000))]
    return np.array(sig, dtype=np.int64)

def _cosim(a: np.ndarray, b: np.ndarray) -> int:
    """Integer cosine similarity × 1000."""
    na = float(np.linalg.norm(a))
    nb = float(np.linalg.norm(b))
    if na < 1e-9 or nb < 1e-9:
        return 1000 if np.array_equal(a, b) else 0
    return int(np.dot(a.astype(float), b.astype(float)) / (na * nb) * 1000)

def _delta(g1: np.ndarray, g2: np.ndarray) -> int:
    """Integer count of changed cells — Noether conservation check."""
    if g1.shape != g2.shape:
        return max(g1.size, g2.size)
    return int(np.count_nonzero(g1 - g2))

def _coord(g: np.ndarray) -> dict:
    """ACTION6 target: minority-value cell (pure integer geometry)."""
    if g.size == 0:
        return {"x": 0, "y": 0}
    vs, cs = np.unique(g, return_counts=True)
    tgt    = int(vs[np.argmin(cs)])
    ys, xs = np.where(g == tgt)
    mid    = len(ys) // 2
    return {"x": int(np.clip(xs[mid], 0, 63)),
            "y": int(np.clip(ys[mid], 0, 63))}

def _avail(fd) -> list:
    try:
        out = []
        for x in (fd.available_actions or []):
            try:
                v = int(x.value)
                out.append("RESET" if v == 0 else f"ACTION{v}")
            except Exception:
                pass
        return out if out else list(_ALL_ACTS)
    except Exception:
        return list(_ALL_ACTS)


class _GHM:
    """Global Harmonic Memory — cross-game Fibonacci-Prime transfer."""
    _inst = None
    def __new__(cls):
        if cls._inst is None:
            o = super().__new__(cls)
            o._wins: list = []
            cls._inst = o
        return cls._inst
    def record(self, sig: np.ndarray, gid: str, seq: list):
        self._wins.append((sig.copy(), gid, list(seq)))
    def top(self, sig: np.ndarray, thr: int = 850) -> list:
        scored = [(_cosim(sig, ws), g, q) for ws, g, q in self._wins]
        scored.sort(reverse=True)
        return [(s, g, q) for s, g, q in scored[:3] if s >= thr]

_ghm = _GHM()


class MyAgent(_AgentBase):
    """
    DEQGT × ANJ ARC-AGI-3 Agent v10 ONE-SHOT FINAL
    ANJ Loop : OBSERVE → ASSIMILATE → EMULATE → TEACH
    Integer-only · No-null · Full-loop · Fibonacci-Prime signatures
    """
    MAX_ACTIONS = _MAX_GAME

    def __init__(self, *args, **kwargs):
        # B5 FIX: accept any signature, pass all args to real base
        gid = (args[0] if args and isinstance(args[0], str)
               else kwargs.get("game_id", "unknown"))
        try:
            super().__init__(*args, **kwargs)
        except TypeError:
            _AgentBase.__init__(self, gid)

        self._phase   = "OBSERVE"
        self._pi      = 0
        self._pr: dict = {}
        self._pq: list = []
        self._bg      = None
        self._bs      = None
        self._pg      = None
        self._pa      = None
        self._seq: list = []
        self._ll      = 0
        self._won     = False
        TRACES.mkdir(parents=True, exist_ok=True)

    def is_done(self, frames, latest_frame) -> bool:
        st = _state(latest_frame)
        if st == "WIN":
            if not self._won and self._bs is not None:
                _ghm.record(self._bs, self.game_id, self._seq)
                self._won = True
            return True
        return st == "GAME_OVER" or self.action_counter >= self.MAX_ACTIONS

    def choose_action(self, frames, latest_frame):
        g  = _grid(latest_frame)
        st = _state(latest_frame)
        lv = int(latest_frame.levels_completed)
        av = _avail(latest_frame)

        # Level transition → reset ANJ phase
        if lv != self._ll:
            self._ll  = lv;  self._phase = "OBSERVE"
            self._pi  = 0;   self._pr    = {}
            self._pq  = [];  self._bg    = None
            self._pg  = None; self._pa   = None

        # Record effect of previous action
        if self._pa is not None and self._pg is not None:
            d = _delta(self._pg, g)
            if self._pa not in self._pr:
                self._pr[self._pa] = d
        self._pg = g.copy()

        # Force RESET on dead / first state
        if st in ("NOT_PLAYED", "GAME_OVER") or self.action_counter == 0:
            self._bg = None;  self._phase = "OBSERVE"
            self._pi = 0;     self._pr    = {}
            self._pq = []
            return self._mk("RESET", g)

        # Capture base grid + GHM lookup after RESET
        if self._bg is None:
            self._bg = g.copy()
            self._bs = _enc(g)
            hits = _ghm.top(self._bs)
            if hits:
                _, src, seq = hits[0]
                self._pq  = list(seq)
                self._phase = "EMULATE"
                print(f"  [{self.game_id}] GHM transfer ← {src} (sim={hits[0][0]})")

        # EMULATE: replay winning sequence from GHM
        if self._phase == "EMULATE" and self._pq:
            a = self._pq.pop(0)
            if a in av:
                return self._mk(a, g)
            self._phase = "OBSERVE"

        # OBSERVE: probe each action once
        if self._phase == "OBSERVE":
            todo = [x for x in av if x != "RESET" and x not in self._pr]
            if todo and self._pi < _PROBE_CAP:
                a = todo[self._pi % len(todo)]
                self._pi += 1
                if self._pi >= len(todo):
                    self._phase = "ASSIMILATE"
                return self._mk(a, g)
            self._phase = "ASSIMILATE"

        # ASSIMILATE: rank by delta, build priority queue
        if self._phase == "ASSIMILATE":
            ranked = sorted(
                [(d, a) for a, d in self._pr.items() if d > 0],
                reverse=True
            )
            self._pq = [a for _, a in ranked]
            if not self._pq:
                self._pq = [x for x in av if x != "RESET"]
            self._phase = "TEACH"

        # TEACH: cycle priority queue (Tesla full-loop — no energy wasted)
        if self._pq:
            for a in list(self._pq):
                if a in av:
                    self._pq.remove(a)
                    self._pq.append(a)
                    return self._mk(a, g)

        # Fallback: round-robin
        nr = [x for x in av if x != "RESET"]
        if nr:
            return self._mk(nr[self.action_counter % len(nr)], g)
        return self._mk("RESET", g)

    def _mk(self, name: str, g: np.ndarray):
        self._pa = name
        self._seq.append(name)
        try:
            act = (GameAction.RESET if name == "RESET"
                   else GameAction.from_name(name))
            if act.is_complex():
                act.set_data(_coord(g))
            act.reasoning = {
                "fw": "DEQGT×ANJv10", "phase": self._phase,
                "step": self.action_counter, "game": self.game_id,
            }
            return act
        except Exception:
            try:
                return GameAction.ACTION1
            except Exception:
                pass

print("      MyAgent ✅")

# ══════════════════════════════════════════════════════════════════════════
# [7] WRITE my_agent.py + .env + register in __init__.py
# ══════════════════════════════════════════════════════════════════════════
print("\n[7/9] Writing agent files …")

_tpl_dir = AGENTS_WK / "agents" / "templates"
_tpl_dir.mkdir(parents=True, exist_ok=True)

_MY_AGENT_CODE = r'''# DEQGT × ANJ ARC-AGI-3 Agent v10 — auto-written by one-shot notebook
import math, os, sys
from pathlib import Path
import numpy as np

_wk = Path("/kaggle/working/ARC-AGI-3-Agents")
if str(_wk) not in sys.path:
    sys.path.insert(0, str(_wk))

from arcengine import GameAction, GameState
from agents.agent import Agent as _Base

_FIB      = [1, 2, 3, 5, 8, 13, 21, 34, 55]
_ALL_ACTS = ["ACTION1","ACTION2","ACTION3","ACTION4","ACTION5","ACTION6","ACTION7"]
_MAX_GAME = 300
_PROBE_CAP = 8

def _g(fd):
    raw = fd.frame[-1] if fd.frame else []
    return np.clip(np.array(raw, dtype=np.int32) if raw
                   else np.zeros((1,1), dtype=np.int32), 0, 15)

def _sn(fd):
    st = fd.state
    return st.name if hasattr(st, "name") else str(st)

def _enc(g, n=9):
    if g.size == 0:
        return np.zeros(n * 2, dtype=np.int64)
    t = g.ravel().astype(float) * (2 * math.pi / 16)
    s = []
    for k in _FIB[:n]:
        c = np.mean(np.exp(1j * k * t))
        s += [int(round(c.real * 1000)), int(round(c.imag * 1000))]
    return np.array(s, dtype=np.int64)

def _cosim(a, b):
    na = float(np.linalg.norm(a)); nb = float(np.linalg.norm(b))
    if na < 1e-9 or nb < 1e-9:
        return 1000 if np.array_equal(a, b) else 0
    return int(np.dot(a.astype(float), b.astype(float)) / (na * nb) * 1000)

def _delta(g1, g2):
    if g1.shape != g2.shape:
        return max(g1.size, g2.size)
    return int(np.count_nonzero(g1 - g2))

def _coord(g):
    if g.size == 0:
        return {"x": 0, "y": 0}
    vs, cs = np.unique(g, return_counts=True)
    tgt = int(vs[np.argmin(cs)])
    ys, xs = np.where(g == tgt)
    mid = len(ys) // 2
    return {"x": int(np.clip(xs[mid], 0, 63)),
            "y": int(np.clip(ys[mid], 0, 63))}

def _av(fd):
    try:
        out = []
        for x in (fd.available_actions or []):
            try:
                v = int(x.value)
                out.append("RESET" if v == 0 else f"ACTION{v}")
            except Exception:
                pass
        return out if out else list(_ALL_ACTS)
    except Exception:
        return list(_ALL_ACTS)

class _GHM:
    _i = None
    def __new__(cls):
        if cls._i is None:
            o = super().__new__(cls); o._w = []; cls._i = o
        return cls._i
    def record(self, sig, gid, seq):
        self._w.append((sig.copy(), gid, list(seq)))
    def top(self, sig, thr=850):
        sc = [(_cosim(sig, ws), g, q) for ws, g, q in self._w]
        sc.sort(reverse=True)
        return [(s, g, q) for s, g, q in sc[:3] if s >= thr]

_ghm = _GHM()

class MyAgent(_Base):
    MAX_ACTIONS = _MAX_GAME
    def __init__(self, *args, **kwargs):
        gid = (args[0] if args and isinstance(args[0], str)
               else kwargs.get("game_id", "?"))
        try:
            super().__init__(*args, **kwargs)
        except TypeError:
            _Base.__init__(self, gid)
        self._phase="OBSERVE"; self._pi=0; self._pr={}
        self._pq=[]; self._bg=None; self._bs=None
        self._pg=None; self._pa=None; self._seq=[]; self._ll=0; self._won=False

    def is_done(self, frames, lf):
        st = _sn(lf)
        if st == "WIN":
            if not self._won and self._bs is not None:
                _ghm.record(self._bs, self.game_id, self._seq)
                self._won = True
            return True
        return st == "GAME_OVER" or self.action_counter >= self.MAX_ACTIONS

    def choose_action(self, frames, lf):
        g = _g(lf); st = _sn(lf)
        lv = int(lf.levels_completed); av = _av(lf)
        if lv != self._ll:
            self._ll=lv; self._phase="OBSERVE"; self._pi=0
            self._pr={}; self._pq=[]; self._bg=None
            self._pg=None; self._pa=None
        if self._pa is not None and self._pg is not None:
            d = _delta(self._pg, g)
            if self._pa not in self._pr:
                self._pr[self._pa] = d
        self._pg = g.copy()
        if st in ("NOT_PLAYED","GAME_OVER") or self.action_counter == 0:
            self._bg=None; self._phase="OBSERVE"; self._pi=0
            self._pr={}; self._pq=[]; return self._mk("RESET", g)
        if self._bg is None:
            self._bg=g.copy(); self._bs=_enc(g)
            hits = _ghm.top(self._bs)
            if hits:
                _, src, seq = hits[0]
                self._pq=list(seq); self._phase="EMULATE"
        if self._phase == "EMULATE" and self._pq:
            a = self._pq.pop(0)
            if a in av: return self._mk(a, g)
            self._phase = "OBSERVE"
        if self._phase == "OBSERVE":
            todo = [x for x in av if x != "RESET" and x not in self._pr]
            if todo and self._pi < _PROBE_CAP:
                a = todo[self._pi % len(todo)]; self._pi += 1
                if self._pi >= len(todo): self._phase = "ASSIMILATE"
                return self._mk(a, g)
            self._phase = "ASSIMILATE"
        if self._phase == "ASSIMILATE":
            ranked = sorted([(d, a) for a, d in self._pr.items() if d > 0], reverse=True)
            self._pq = [a for _, a in ranked]
            if not self._pq: self._pq = [x for x in av if x != "RESET"]
            self._phase = "TEACH"
        if self._pq:
            for a in list(self._pq):
                if a in av:
                    self._pq.remove(a); self._pq.append(a)
                    return self._mk(a, g)
        nr = [x for x in av if x != "RESET"]
        if nr: return self._mk(nr[self.action_counter % len(nr)], g)
        return self._mk("RESET", g)

    def _mk(self, name, g):
        self._pa = name; self._seq.append(name)
        try:
            act = (GameAction.RESET if name == "RESET"
                   else GameAction.from_name(name))
            if act.is_complex(): act.set_data(_coord(g))
            act.reasoning = {"fw":"DEQGTv10","ph":self._phase,
                             "n":self.action_counter,"gid":self.game_id}
            return act
        except Exception:
            try: return GameAction.ACTION1
            except Exception: pass
'''

(_tpl_dir / "my_agent.py").write_text(_MY_AGENT_CODE, encoding="utf-8")
print("      templates/my_agent.py ✅")

_init_path.write_text(
'''# DEQGT-patched __init__ — no langgraph / no smolagents
from typing import Type
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.my_agent import MyAgent
try:
    from .templates.random_agent import Random
    AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
        "random": Random, "myagent": MyAgent
    }
except ImportError:
    AVAILABLE_AGENTS: dict[str, Type[Agent]] = {"myagent": MyAgent}
''', encoding="utf-8"
)
print("      agents/__init__.py → MyAgent registered ✅")

(AGENTS_WK / ".env").write_text(
    "SCHEME=http\nHOST=gateway\nPORT=8001\n"
    "ARC_API_KEY=test-key-123\n"
    "ARC_BASE_URL=http://gateway:8001/\n"
    "OPERATION_MODE=online\n"
    "ENVIRONMENTS_DIR=\n"
    "RECORDINGS_DIR=/kaggle/working/server_recording\n",
    encoding="utf-8"
)
print("      .env ✅")

# ══════════════════════════════════════════════════════════════════════════
# [8] WRITE FINAL submission.parquet (offline placeholder rows)
#     Real score comes from gateway competition rerun — this just satisfies
#     Kaggle's "output file must exist" requirement.
# ══════════════════════════════════════════════════════════════════════════
print("\n[8/9] Writing final submission.parquet …")
rows = []
for gid in GAME_IDS:
    for i in range(10):
        rows.append({"row_id": f"{gid}_{i}", "game_id": gid,
                     "end_of_game": False, "score": 1})
    rows.append({"row_id": f"{gid}_10", "game_id": gid,
                 "end_of_game": True, "score": 1})

df = pd.DataFrame(rows, columns=["row_id","game_id","end_of_game","score"])
df.to_parquet(str(PARQUET), index=False)
sz = PARQUET.stat().st_size

print(f"\n{'═'*65}")
print(f"  ✅  submission.parquet  FINAL WRITTEN")
print(f"      Path  : {PARQUET}")
print(f"      Rows  : {len(df)}")
print(f"      Games : {len(GAME_IDS)}")
print(f"      Size  : {sz:,} bytes")
print(df.tail(4).to_string(index=False))
print(f"{'═'*65}")

# ══════════════════════════════════════════════════════════════════════════
# [9] COMPETITION RERUN → gateway → 110 games → real score
# ══════════════════════════════════════════════════════════════════════════
if IS_RERUN:
    print("\n" + "═"*65)
    print("  COMPETITION RERUN ACTIVE — connecting to live gateway …")
    print("═"*65)
    os.system(
        "curl --silent --retry 30 --retry-delay 3 "
        "--retry-connrefused http://gateway:8001/api/games || true"
    )
    time.sleep(2)
    ret = os.system(
        f"cd {AGENTS_WK} && MPLBACKEND=agg python main.py --agent myagent"
    )
    print(f"\n  Gateway swarm exit code: {ret}")
    print("  ✅ Gateway run complete." if ret == 0
          else "  [WARN] check gateway logs above")
else:
    print("\n" + "─"*65)
    print("  OFFLINE RUN COMPLETE — submission.parquet READY")
    print("")
    print("  ┌─────────────────────────────────────────────────────┐")
    print("  │  HOW TO SUBMIT (do this EXACTLY):                   │")
    print("  │                                                     │")
    print("  │  1. Click  Save Version  (top right)                │")
    print("  │  2. Choose  Save & Run All (Commit)   ← CRITICAL    │")
    print("  │     NOT Quick Save — NOT Draft Session              │")
    print("  │  3. Wait for green checkmark  (~3 min)              │")
    print("  │  4. ARC Prize 2026 → Submit Prediction              │")
    print("  │  5. Pick THIS notebook + the new COMMITTED version  │")
    print("  │  6. Click Submit                                    │")
    print("  └─────────────────────────────────────────────────────┘")
    print("─"*65)
